In [2]:
"""
11_club_scouting_interactive.py -- ScoutAI Director (Opportunity Mode)

An interactive terminal interface that allows club scouts or directors to 
filter players by position, budget, and age criteria, rank them using 
custom performance metrics, and surface the highest-value transfer gems.
"""

import os
import sys
import pandas as pd
import numpy as np
import sqlalchemy
import joblib
from dotenv import load_dotenv, find_dotenv

# ==========================================
# 0. CONFIG & SETUP
# ==========================================

load_dotenv(find_dotenv())

MODELS_DIR = "models"
DATA_DIR = "data"

os.makedirs(DATA_DIR, exist_ok=True)

LEAGUE_WEIGHTS = {
    'Premier League': 1.5, 'LaLiga': 1.4, 'Serie A': 1.3, 'Bundesliga': 1.3,
    'Ligue 1': 1.2, 'Eredivisie': 1.15, 'Liga Portugal': 1.15, 'Süper Lig': 1.05,
}

FULL_FEATURES = [
    'age', 'height_in_cm', 'total_appearances', 'minutes_per_match',
    'total_goals', 'total_assists', 'goals_per_90', 'assists_per_90',
    'total_yellow_cards', 'total_red_cards', 'international_caps', 'international_goals',
    'club_squad_size', 'club_avg_age', 'stadium_seats', 'foreigners_percentage',
    'contract_months_remaining', 'wonderkid_hype', 'league_coefficient',
    'has_transfer_history', 'max_career_transfer_fee', 'most_recent_transfer_fee',
    'detailed_position', 'foot', 'passport_tier',
]

PERFORMANCE_FEATURES = [
    'age', 'height_in_cm', 'total_appearances', 'minutes_per_match',
    'total_goals', 'total_assists', 'goals_per_90', 'assists_per_90',
    'total_yellow_cards', 'total_red_cards', 'international_caps', 'international_goals',
    'club_squad_size', 'club_avg_age', 'stadium_seats', 'foreigners_percentage',
    'wonderkid_hype', 'league_coefficient', 'detailed_position', 'foot', 'passport_tier',
]


# ==========================================
# 1. DATA PREPARATION & PREDICTION ROUTING
# ==========================================

def load_and_prepare_data(engine):
    df = pd.read_sql("SELECT * FROM view_scout_master", engine)

    numeric_cols = [
        'age', 'total_appearances', 'international_caps', 'international_goals',
        'passport_power_rank', 'height_in_cm', 'minutes_per_match', 'total_goals',
        'total_assists', 'goals_per_90', 'assists_per_90', 'total_yellow_cards',
        'total_red_cards', 'club_squad_size', 'club_avg_age', 'stadium_seats',
        'foreigners_percentage', 'contract_months_remaining', 'max_career_transfer_fee',
        'most_recent_transfer_fee', 'current_market_value',
    ]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

    df['wonderkid_hype'] = np.where(df['age'] <= 22, (df['total_appearances'] + (df['international_caps'] * 3)), 0)

    conditions = [
        (df['passport_power_rank'].fillna(999) <= 15),
        (df['passport_power_rank'].fillna(999) > 15) & (df['passport_power_rank'].fillna(999) <= 60),
    ]
    df['passport_tier'] = np.select(conditions, ['Tier_1', 'Tier_2'], default='Tier_3')
    df['league_coefficient'] = df['competition_name'].map(LEAGUE_WEIGHTS).fillna(1.0)
    df['detailed_position'] = df['sub_position'].fillna(df['position_group']).astype(str)

    return df


def add_model_predictions(df, model_full, model_perf):
    df = df.copy()
    df['predicted_value'] = np.nan

    has_history_mask = df['has_transfer_history'] == 1
    no_history_mask = ~has_history_mask

    if has_history_mask.any():
        log_preds = model_full.predict(df.loc[has_history_mask, FULL_FEATURES])
        df.loc[has_history_mask, 'predicted_value'] = np.expm1(log_preds)

    if no_history_mask.any():
        log_preds = model_perf.predict(df.loc[no_history_mask, PERFORMANCE_FEATURES])
        df.loc[no_history_mask, 'predicted_value'] = np.expm1(log_preds)

    return df


# ==========================================
# 2. RECOMMENDATION CORE LOGIC
# ==========================================

def get_club_recommendations(position, min_budget, max_budget, min_age, max_age, primary_metric,
                             secondary_metric, top_n, df):
    df = df.copy()

    df['search_pos'] = df['position_group'].astype(str).fillna('') + " " + df['sub_position'].astype(str).fillna('')
    df['search_pos'] = df['search_pos'].str.lower().str.replace("-", " ")

    clean_pos_input = position.replace('İ', 'i').replace('I', 'ı').lower().replace("-", " ").strip()

    # Filtering with both Min and Max Budget
    pos_match_df = df[df['search_pos'].str.contains(clean_pos_input, case=False, na=False)]
    budget_match_df = pos_match_df[(pos_match_df['current_market_value'] >= min_budget) & (pos_match_df['current_market_value'] <= max_budget)]
    filtered_df = budget_match_df[(budget_match_df['age'] >= min_age) & (budget_match_df['age'] <= max_age)].copy()

    if filtered_df.empty:
        print("\n[WARNING] No players found matching your criteria.")
        return None

    for col in [primary_metric, secondary_metric]:
        col_to_use = col if col in filtered_df.columns else 'total_appearances'
        filtered_df[col_to_use] = pd.to_numeric(filtered_df[col_to_use], errors='coerce').fillna(0)
        rank = filtered_df[col_to_use].rank(pct=True)
        filtered_df[f'norm_{col}'] = 1 / (1 + np.exp(-10 * (rank - 0.5)))

    filtered_df['value_gap'] = (filtered_df['predicted_value'] - filtered_df['current_market_value']) / (filtered_df['current_market_value'] + 1)
    filtered_df['abs_value_gap'] = filtered_df['predicted_value'] - filtered_df['current_market_value']
    filtered_df['pct_gap_score'] = filtered_df['value_gap'].rank(pct=True)

    abs_gap_min = filtered_df['abs_value_gap'].min()
    abs_gap_max = filtered_df['abs_value_gap'].max()
    abs_gap_range = abs_gap_max - abs_gap_min
    if abs_gap_range > 0:
        filtered_df['abs_gap_score'] = (filtered_df['abs_value_gap'] - abs_gap_min) / abs_gap_range
    else:
        filtered_df['abs_gap_score'] = 0.0

    BLEND_ABS_WEIGHT = 0.65
    filtered_df['gap_score'] = (
        (1 - BLEND_ABS_WEIGHT) * filtered_df['pct_gap_score']
        + BLEND_ABS_WEIGHT * filtered_df['abs_gap_score']
    )

    filtered_df['Custom_Scout_Score'] = (
        (filtered_df[f'norm_{primary_metric}'] * 0.4) +
        (filtered_df[f'norm_{secondary_metric}'] * 0.3) +
        (filtered_df['gap_score'] * 0.3)
    )

    recommendations = filtered_df.sort_values(by=['Custom_Scout_Score'], ascending=False).head(top_n)

    print(f"\n{'='*80}\n 🎯 SCOUT AI: OPPORTUNITY LIST (Top {len(recommendations)} Matches)\n{'='*80}")

    txt_export_path = os.path.join(DATA_DIR, "transfer_list.txt")
    with open(txt_export_path, "w", encoding="utf-8") as f:
        f.write("ScoutAI Custom Transfer Target List\n=========================================\n")
        for _, player in recommendations.iterrows():
            pos = player['sub_position'] if pd.notna(player['sub_position']) else player['position_group']
            line = (
                f"{player['player_name']} | {pos} | {player['club_name']} | "
                f"Value: €{float(player['current_market_value']):,.0f} | "
                f"Model ({player['model_used']}): €{float(player['predicted_value']):,.0f} | "
                f"Gap: {float(player['value_gap'])*100:.1f}% | "
                f"Score: {float(player['Custom_Scout_Score'])*100:.1f}"
            )
            print(line)
            f.write(line + "\n")

    print(f"\n[SYSTEM] {len(recommendations)} targets identified and exported to '{txt_export_path}'.")
    return recommendations


# ==========================================
# INTERACTIVE TERMINAL UTILITIES
# ==========================================

METRIC_MAP = {
    "goals": "total_goals", "assists": "total_assists", "appearances": "total_appearances",
    "minutes played": "minutes_per_match", "international caps": "international_caps",
    "international goals": "international_goals", "goals per 90": "goals_per_90",
    "assists per 90": "assists_per_90", "contract remaining": "contract_months_remaining",
}

POSITION_DEFAULT_METRICS = [
    (["goalkeeper", "gk"], "appearances", "contract remaining"),
    (["back", "centre-back", "center-back", "cb", "sweeper"], "appearances", "international caps"),
    (["wing-back", "wingback"], "assists", "appearances"),
    (["defensive midfield", "dm"], "appearances", "contract remaining"),
    (["midfield"], "assists", "appearances"),
    (["winger", "wing"], "goals", "assists"),
    (["forward", "striker", "centre-forward", "center-forward", "cf"], "goals", "goals per 90"),
]
DEFAULT_FALLBACK = ("goals", "assists")


def suggest_defaults_for_position(position: str):
    pos_lower = position.lower()
    for keywords, primary, secondary in POSITION_DEFAULT_METRICS:
        if any(kw in pos_lower for kw in keywords):
            return primary, secondary
    return DEFAULT_FALLBACK


def get_valid_int(prompt):
    while True:
        try:
            return int(input(prompt).strip())
        except ValueError:
            print("[ERROR] Invalid number.")


def get_valid_float(prompt):
    while True:
        try:
            return float(input(prompt).strip())
        except ValueError:
            print("[ERROR] Invalid number.")


def get_valid_metric(prompt, default_key):
    while True:
        user_input = input(f"{prompt} [Press Enter for '{default_key}']: ").strip().lower()
        if user_input == "":
            return METRIC_MAP[default_key]
        if user_input in METRIC_MAP:
            return METRIC_MAP[user_input]
        print(f"\n[ERROR] Available options: {', '.join(METRIC_MAP.keys())}\n")


# ==========================================
# MAIN EXECUTION
# ==========================================

def main():
    print("\n" + "*"*60 + "\n 🕵️‍♂️ WELCOME TO SCOUT AI DIRECTOR - OPPORTUNITY MODE\n" + "*"*60)

    db_url = os.getenv('DB_URL')
    if not db_url:
        print("[ERROR] DB_URL not found. Please ensure your .env file exists and is configured correctly.")
        sys.exit(1)

    print("[SYSTEM] Connecting to database and loading models...")
    try:
        engine = sqlalchemy.create_engine(db_url)
        model_full = joblib.load(os.path.join(MODELS_DIR, 'scout_model_full.pkl'))
        model_perf = joblib.load(os.path.join(MODELS_DIR, 'scout_model_performance_only.pkl'))
    except FileNotFoundError:
        print(f"[ERROR] Required models not found in '{MODELS_DIR}'. Run the pipeline scripts first.")
        sys.exit(1)
    except Exception as e:
        print(f"[ERROR] System initialization failed: {e}")
        sys.exit(1)

    df_raw = load_and_prepare_data(engine)
    has_history_mask = df_raw['has_transfer_history'] == 1
    df_raw['model_used'] = np.where(has_history_mask, 'full', 'performance_only')
    df = add_model_predictions(df_raw, model_full, model_perf)

    print("\n--- ScoutAI Director Terminal Ready ---")
    
    try:
        u_pos = input("\n1. Position (e.g., Striker, Goalkeeper): ").strip()
        if not u_pos: 
            print("[SYSTEM] No position entered. Exiting.")
            sys.exit(0)

        u_min_bud = get_valid_float("2. Min Budget in € (Type 0 for no minimum): ")
        u_max_bud = get_valid_float("3. Max Budget in €: ")
        u_min = get_valid_int("4. Min Age: ")
        u_max = get_valid_int("5. Max Age: ")

        suggested_primary, suggested_secondary = suggest_defaults_for_position(u_pos)
        print(f"\n[SYSTEM] Suggested metrics for '{u_pos}': primary='{suggested_primary}', secondary='{suggested_secondary}'")
        print(f"Metrics: {', '.join(METRIC_MAP.keys())}")
        
        u_m1 = get_valid_metric("6. Primary Metric", suggested_primary)
        u_m2 = get_valid_metric("7. Secondary Metric", suggested_secondary)
        u_top = get_valid_int("8. Number of players to view: ")

        get_club_recommendations(u_pos, u_min_bud, u_max_bud, u_min, u_max, u_m1, u_m2, u_top, df)

    except KeyboardInterrupt:
        print("\n[SYSTEM] Action cancelled.")
    except Exception as e:
        print(f"\n[ERROR] An error occurred during lookup: {e}")

    print("\nExiting ScoutAI Director. Good luck with the transfer window!")

if __name__ == "__main__":
    main()


************************************************************
 🕵️‍♂️ WELCOME TO SCOUT AI DIRECTOR - OPPORTUNITY MODE
************************************************************
[SYSTEM] Connecting to database and loading models...

--- ScoutAI Director Terminal Ready ---

[SYSTEM] Suggested metrics for 'right winger': primary='goals', secondary='assists'
Metrics: goals, assists, appearances, minutes played, international caps, international goals, goals per 90, assists per 90, contract remaining

 🎯 SCOUT AI: OPPORTUNITY LIST (Top 5 Matches)
Rodrygo | Right Winger | Real Madrid Club de Fútbol | Value: €45,000,000 | Model (full): €63,510,804 | Gap: 41.1% | Score: 98.7
Ritsu Doan | Right Winger | Eintracht Frankfurt Fußball AG | Value: €17,000,000 | Model (full): €23,869,712 | Gap: 40.4% | Score: 93.3
Moussa Diaby | Right Winger | Al-Ittihad Football Club | Value: €25,000,000 | Model (full): €28,486,852 | Gap: 13.9% | Score: 90.6
Dodi Lukébakio | Right Winger | Sport Lisboa e Benfica | 